# 全参考评估脚本：PSNR / SSIM / LPIPS / 线条 F1

本 notebook 用于在 **同一测试集** 上，对不同模型的 **配对 SR/HR 图像** 做全参考评估：

- 失真类指标：**PSNR / SSIM**（基于像素与结构的还原度）
- 感知类指标：**LPIPS** （越低越好，1-LPIPS 越高越好）
- 动漫特化指标：**线条保真度 F1**（Canny 边缘检测 + F1-score）

已适配以下架构：
- 我们的 RRDB 23-block / 6-block（2× / 4×）
- APISR 官方 RRDB 2x / 4x 预训练模型

本 notebook 只做 **评估**，不进行训练，默认在本地/Colab 已经生成好 SR 图像或已准备好 LR/HR 及模型权重。

In [ ]:
# ============================================================
# 0. 基本配置
# ============================================================

import os
import glob
import json
from math import log10

import cv2
import numpy as np
from tqdm import tqdm

import torch
import torch.nn.functional as F

from skimage.metrics import structural_similarity as ssim
import lpips

from architecture import build_rrdb

# 设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# 根据需要修改下面这组配置来选择要评估的模型
MODEL_NAME   = 'ESRGAN_4x_23block_simple'   # 结果保存时使用的标签
SCALE        = 4                            # 2 或 4
NUM_BLOCK    = 23                           # 6 或 23
IS_APISR     = False                        # True: 加载 APISR 官方 RRDB; False: 加载我们自己的 checkpoint

# 我们自己的 checkpoint（例如 Colab 训练出来的 best 模型）
CKPT_PATH_OURS = 'saved_models/ESRGAN_4x_23block_simple_best.pth'

# APISR 官方 RRDB 预训练权重（放在 pretrained_models/ 下）
CKPT_PATH_APISR_2X = 'pretrained_models/2x_APISR_RRDB_GAN_generator.pth'
CKPT_PATH_APISR_4X = 'pretrained_models/4x_APISR_RRDB_GAN_generator.pth'

# 成对的 LR/HR 数据集
LR_DIR = 'dataset/lowres_4x_simple/original'  # 或 dataset/lowres_4x/original, dataset/lowres_2x/original 等
HR_DIR = 'dataset/highres/original'

# 若已有现成 SR 结果，也可以直接评估 SR vs HR，而不重新推理
USE_EXISTING_SR = False
SR_DIR_EXISTING = 'results/ESRGAN_4x_23block_simple_inference'

# SR 输出目录（若需要重新推理）
SR_DIR = f'results/{MODEL_NAME}_eval_sr'
os.makedirs(SR_DIR, exist_ok=True)

# 结果输出路径（若在 Colab 的环境初始化 notebook 中创建了软链接，results 会指向 Google Drive）
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
# ============================================================
# 1. 构建并加载模型
# ============================================================

def load_rrdb_model(model_name: str, scale: int, num_block: int, is_apisr: bool):
    model = build_rrdb(scale=scale, num_block=num_block).to(device)
    model.eval()

    if is_apisr:
        ckpt_path = CKPT_PATH_APISR_2X if scale == 2 else CKPT_PATH_APISR_4X
        print(f'加载 APISR RRDB 权重: {ckpt_path}')
        ckpt = torch.load(ckpt_path, map_location=device)
        if 'params_ema' in ckpt:
            state = ckpt['params_ema']
        elif 'model_state_dict' in ckpt:
            state = ckpt['model_state_dict']
        else:
            state = ckpt
        model_state = model.state_dict()
        compatible = {k: v for k, v in state.items() if k in model_state and model_state[k].shape == v.shape}
        model_state.update(compatible)
        model.load_state_dict(model_state, strict=False)
        print(f'  兼容加载参数: {len(compatible)} / {len(model_state)}')
    else:
        ckpt_path = CKPT_PATH_OURS
        print(f'加载自训练权重: {ckpt_path}')
        ckpt = torch.load(ckpt_path, map_location=device)
        if 'model_state_dict' in ckpt:
            state = ckpt['model_state_dict']
        elif 'params_ema' in ckpt:
            state = ckpt['params_ema']
        else:
            state = ckpt
        model.load_state_dict(state, strict=False)
        print(f"  Epoch: {ckpt.get('epoch', '?')}, Loss: {ckpt.get('loss', '?')}")

    return model


model = load_rrdb_model(MODEL_NAME, SCALE, NUM_BLOCK, IS_APISR)


In [ ]:
# ============================================================
# 2. 推理函数（支持 2x/4x + 大图分块）
# ============================================================

def sr_infer_single(lr_img_bgr: np.ndarray, scale: int) -> np.ndarray:
    """对单张 LR 图像做超分，返回 RGB uint8 图像。"""
    lr_img = cv2.cvtColor(lr_img_bgr, cv2.COLOR_BGR2RGB)
    h, w = lr_img.shape[:2]

    # 2x RRDB 有 pixel_unshuffle 要求输入尺寸能被 4 整除
    pad_h = (4 - h % 4) % 4 if SCALE == 2 else 0
    pad_w = (4 - w % 4) % 4 if SCALE == 2 else 0
    lr_padded = lr_img
    if pad_h > 0 or pad_w > 0:
        lr_padded = cv2.copyMakeBorder(lr_img, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)

    lr_t = torch.from_numpy(lr_padded.transpose(2, 0, 1)).float().unsqueeze(0).to(device) / 255.0

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            _, _, hh, ww = lr_t.shape
            if hh * ww > 512 * 512:
                h_half, w_half = hh // 2, ww // 2
                sr_t = torch.zeros((1, 3, hh * scale, ww * scale), device=device)
                sr_t[:, :, :h_half*scale, :w_half*scale] = model(lr_t[:, :, :h_half, :w_half])
                sr_t[:, :, :h_half*scale, w_half*scale:] = model(lr_t[:, :, :h_half, w_half:])
                sr_t[:, :, h_half*scale:, :w_half*scale] = model(lr_t[:, :, h_half:, :w_half])
                sr_t[:, :, h_half*scale:, w_half*scale:] = model(lr_t[:, :, h_half:, w_half:])
            else:
                sr_t = model(lr_t)

    sr = sr_t.squeeze(0).float().cpu().clamp(0, 1).numpy().transpose(1, 2, 0)
    # 裁掉 padding 对应的区域
    sr = sr[:h * scale, :w * scale, :]
    sr_uint8 = (sr * 255.0).round().astype(np.uint8)
    return sr_uint8


def ensure_dir(path: str):
    if path and not os.path.isdir(path):
        os.makedirs(path, exist_ok=True)


ensure_dir(SR_DIR)

# 收集成对的 LR/HR 路径
lr_paths = sorted(glob.glob(os.path.join(LR_DIR, '*.[jp][pn]*g')))
print(f'找到 LR 图像: {len(lr_paths)} 张')


In [ ]:
# ============================================================
# 3. 计算全参考指标：PSNR / SSIM / LPIPS / 线条 F1
# ============================================================

lpips_model = lpips.LPIPS(net='vgg').to(device)
lpips_model.eval()

def img_to_tensor_lpips(img_rgb: np.ndarray) -> torch.Tensor:
    """将 [0,255] RGB 转为 LPIPS 所需的 [-1,1] NCHW tensor"""
    x = img_rgb.astype(np.float32) / 255.0
    x = x * 2.0 - 1.0
    x = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)
    return x.to(device)


def canny_f1(sr_gray: np.ndarray, hr_gray: np.ndarray, t1: int = 100, t2: int = 200) -> float:
    """基于 Canny 边缘的 F1 指标。输入为 uint8 灰度图。"""
    sr_edges = cv2.Canny(sr_gray, t1, t2)
    hr_edges = cv2.Canny(hr_gray, t1, t2)
    sr_bin = sr_edges > 0
    hr_bin = hr_edges > 0
    tp = np.logical_and(sr_bin, hr_bin).sum()
    fp = np.logical_and(sr_bin, ~hr_bin).sum()
    fn = np.logical_and(~sr_bin, hr_bin).sum()
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return float(f1)


psnr_list, ssim_list, lpips_list, f1_list = [], [], [], []
per_image_scores = {}

for lr_path in tqdm(lr_paths, desc='Eval (SR vs HR)'):
    fname = os.path.basename(lr_path)
    hr_path = None
    stem, _ext = os.path.splitext(fname)
    # HR 可能是 png 或 jpg
    for ext in ['.png', '.jpg', '.jpeg', '.JPG', '.JPEG']:
        cand = os.path.join(HR_DIR, stem + ext)
        if os.path.isfile(cand):
            hr_path = cand
            break
    if hr_path is None:
        continue

    hr_bgr = cv2.imread(hr_path)
    if hr_bgr is None:
        continue
    hr_rgb = cv2.cvtColor(hr_bgr, cv2.COLOR_BGR2RGB)

    # 获取/生成 SR
    if USE_EXISTING_SR:
        sr_path = os.path.join(SR_DIR_EXISTING, fname)
        sr_bgr = cv2.imread(sr_path)
        if sr_bgr is None:
            continue
        sr_rgb = cv2.cvtColor(sr_bgr, cv2.COLOR_BGR2RGB)
    else:
        lr_bgr = cv2.imread(lr_path)
        if lr_bgr is None:
            continue
        sr_rgb = sr_infer_single(lr_bgr, SCALE)
        # 保存 SR
        cv2.imwrite(os.path.join(SR_DIR, fname), cv2.cvtColor(sr_rgb, cv2.COLOR_RGB2BGR))

    # 尺寸对齐（若有轻微差异）
    h_hr, w_hr = hr_rgb.shape[:2]
    h_sr, w_sr = sr_rgb.shape[:2]
    if (h_hr != h_sr) or (w_hr != w_sr):
        sr_rgb = cv2.resize(sr_rgb, (w_hr, h_hr), interpolation=cv2.INTER_CUBIC)

    # 转 float32 [0,1]
    hr_f = hr_rgb.astype(np.float32) / 255.0
    sr_f = sr_rgb.astype(np.float32) / 255.0

    # PSNR
    mse = np.mean((hr_f - sr_f) ** 2)
    if mse == 0:
        psnr_val = float('inf')
    else:
        psnr_val = 10 * log10(1.0 / mse)

    # SSIM（skimage 期望灰度或多通道，data_range=1.0）
    ssim_val = ssim(hr_f, sr_f, channel_axis=2, data_range=1.0)

    # LPIPS（越低越好）
    hr_lp = img_to_tensor_lpips(hr_rgb)
    sr_lp = img_to_tensor_lpips(sr_rgb)
    with torch.no_grad():
        lpips_val = float(lpips_model(hr_lp, sr_lp).item())

    # 线条 F1（基于灰度图的 Canny 边缘）
    hr_gray = cv2.cvtColor(hr_rgb, cv2.COLOR_RGB2GRAY)
    sr_gray = cv2.cvtColor(sr_rgb, cv2.COLOR_RGB2GRAY)
    f1_val = canny_f1(sr_gray, hr_gray)

    psnr_list.append(psnr_val)
    ssim_list.append(ssim_val)
    lpips_list.append(lpips_val)
    f1_list.append(f1_val)

    per_image_scores[fname] = {
        'PSNR': round(psnr_val, 4),
        'SSIM': round(ssim_val, 4),
        'LPIPS': round(lpips_val, 4),
        'EdgeF1': round(f1_val, 4),
    }

print('评估完成，共有效样本:', len(psnr_list))


In [ ]:
# ============================================================
# 4. 统计 & 持久化输出
# ============================================================

if len(psnr_list) == 0:
    print('⚠ 未得到任何有效样本，请检查 LR_DIR / HR_DIR / 路径配置。')
else:
    avg_psnr  = float(np.mean(psnr_list))
    avg_ssim  = float(np.mean(ssim_list))
    avg_lpips = float(np.mean(lpips_list))
    avg_f1    = float(np.mean(f1_list))

    print('\n' + '='*60)
    print(f'  {MODEL_NAME} 全参考评估结果')
    print('='*60)
    print(f'  PSNR   (↑, dB)     : {avg_psnr:.4f}')
    print(f'  SSIM   (↑)         : {avg_ssim:.4f}')
    print(f'  LPIPS  (↓, VGG)    : {avg_lpips:.4f}')
    print(f'  Edge F1 (↑, Canny) : {avg_f1:.4f}')
    print('='*60)

    summary = {
        'model': MODEL_NAME,
        'scale': SCALE,
        'num_block': NUM_BLOCK,
        'is_apisr': IS_APISR,
        'num_images': len(psnr_list),
        'metrics': {
            'PSNR': round(avg_psnr, 4),
            'SSIM': round(avg_ssim, 4),
            'LPIPS': round(avg_lpips, 4),
            'EdgeF1': round(avg_f1, 4),
        },
        'per_image': per_image_scores,
    }

    json_path = os.path.join(RESULTS_DIR, f'{MODEL_NAME}_fullref_eval.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f'JSON 结果已保存: {json_path}')

    txt_path = os.path.join(RESULTS_DIR, f'{MODEL_NAME}_fullref_eval.txt')
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(f'{MODEL_NAME} (scale={SCALE}x, blocks={NUM_BLOCK}, is_apisr={IS_APISR})\n')
        f.write(f'num_images = {len(psnr_list)}\n')
        f.write('='*40 + '\n')
        f.write(f'PSNR   (↑, dB)     : {avg_psnr:.4f}\n')
        f.write(f'SSIM   (↑)         : {avg_ssim:.4f}\n')
        f.write(f'LPIPS  (↓, VGG)    : {avg_lpips:.4f}\n')
        f.write(f'Edge F1 (↑, Canny) : {avg_f1:.4f}\n')
        f.write('='*40 + '\n')
    print(f'TXT 结果已保存: {txt_path}')
    print('\n✅ 全参考评估完成，结果已持久化。')
